In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'),
(107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'),
(108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed');

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES(109,'Rahul Verma','Mumbai','Tablet',1,27000,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES 
(110,'Arjun Nair','kochi','Mobile',1,29000,'Placed'),
(111,'Kavya Reddy','Hyderabad','Laptop',1,78000,'Placed');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES(112,'Neha Kappor','Pune','Smartwatch',1,15000,'Shipped');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES(113,'Suresh Rao','Delhi','Headphones',5,2500,'Placed');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(114,'Ramesh Gupta','Hyderabad','Mobile',1,26000,'Placed'),
(115,'Pooja Singh','Hyderabad','Tablet',2,24000,'Placed'),
(116,'Nikhil Rao','Hyderabad','Laptop',1,77000,'Placed');


num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
UPDATE ecommerce_orders SET order_status='Shipped' WHERE order_id=101;

num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET quantity=quantity+1 WHERE order_id=102;

num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET price=78000 WHERE product='Laptop';

num_affected_rows
5


In [0]:
%sql
UPDATE ecommerce_orders SET city='Secunderabad' WHERE customer_name='Amit Sharma';

num_affected_rows
1


In [0]:
%sql
UPDATE ecommerce_orders SET order_status='Delivered' WHERE product='Mobile';

num_affected_rows
4


In [0]:
%sql
UPDATE ecommerce_orders SET price=2500 WHERE product='Headphones';

num_affected_rows
2


In [0]:
%sql
UPDATE ecommerce_orders SET  price=price+1000 WHERE product='Tablet';

num_affected_rows
3


In [0]:
%sql
UPDATE ecommerce_orders SET order_status='Processing' WHERE city='Bangalore';

num_affected_rows
2


In [0]:
%sql
UPDATE ecommerce_orders SET quantity=2 WHERE quantity=1;

num_affected_rows
11


In [0]:
%sql
UPDATE ecommerce_orders SET city='Surat' WHERE city='Ahmedabad';

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders WHERE order_status='Cancelled';

num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders WHERE quantity>3;

num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders WHERE product='Camera';

num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders WHERE city='Kolkata';

num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders WHERE price<5000;

num_affected_rows
1


In [0]:
DELETE FROM ecommerce_orders WHERE customer_name LIKE 'A%';

num_affected_rows
2


In [0]:
DELETE FROM ecommerce_orders WHERE product='Tablet';

num_affected_rows
2


In [0]:
DELETE FROM ecommerce_orders WHERE city='Mumbai' AND quantity=1;

num_affected_rows
0


In [0]:
DELETE FROM ecommerce_orders WHERE price>80000;

num_affected_rows
0


In [0]:
DELETE FROM ecommerce_orders WHERE order_id=(SELECT MAX(order_id) FROM ecommerce_orders);

num_affected_rows
1


In [0]:
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')
AS incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id=source.order_id
WHEN MATCHED THEN
UPDATE SET
target.customer_name=source.customer_name,
target.city=source.city,
target.product=source.product,
target.quantity=source.quantity,
target.price=source.price,
target.order_status=source.order_status
WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2


In [0]:
SELECT * FROM ecommerce_orders ORDER BY order_id;

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
112,Neha Kappor,Pune,Smartwatch,2,15000,Shipped
114,Ramesh Gupta,Hyderabad,Mobile,2,26000,Delivered


In [0]:
SELECT CASE WHEN target.order_id IS NOT NULL THEN 'Updated' ELSE 'Inserted' END AS merge_action,source.* FROM incoming_orders AS source LEFT JOIN ecommerce_orders AS target ON source.order_id=target.order_id;

merge_action,order_id,customer_name,city,product,quantity,price,order_status
Updated,102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
Updated,104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
Updated,109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
Updated,110,Divya Menon,Kochi,Mobile,1,27000,Placed
Updated,111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed


In [0]:
MERGE INTO ecommerce_orders t USING incoming_orders s ON t.order_id=s.order_id WHEN MATCHED THEN UPDATE SET t.order_status=s.order_status;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
MERGE INTO ecommerce_orders t USING incoming_orders s ON t.order_id=s.order_id WHEN MATCHED AND s.quantity>t.quantity THEN UPDATE SET t.quantity=s.quantity;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
MERGE INTO ecommerce_orders t USING(SELECT * FROM incoming_orders WHERE order_status!='Cancelled') s ON t.order_id=s.order_id WHEN MATCHED THEN UPDATE SET* WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
MERGE INTO ecommerce_orders t USING (SELECT*from incoming_orders WHERE product='Laptop') s ON t.order_id=s.order_id WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
MERGE INTO ecommerce_orders t USING incoming_orders s ON t.order_id=s.order_id WHEN MATCHED AND s.price>t.price THEN UPDATE SET t.price=s.price;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'),
(106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'),
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed')
AS daily_updates(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
MERGE INTO ecommerce_orders t
USING daily_updates s
ON t.order_id=s.order_id
WHEN MATCHED THEN
UPDATE SET
t.quantity=s.quantity,
t.price=s.price,
t.order_status=s.order_status
WHEN NOT MATCHED THEN
INSERT *;


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,1,0,2


In [0]:
SELECT COUNT(*) AS updated_rows FROM ecommerce_orders WHERE order_id IN (103,106);

updated_rows
2


In [0]:
SELECT COUNT(*) AS inserted_rows FROM ecommerce_orders WHERE order_id=112;

inserted_rows
1


In [0]:
SELECT * FROM ecommerce_orders ORDER BY price DESC;

order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,2,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
106,Ananya Das,Kolkata,Mobile,2,28000,Shipped
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
114,Ramesh Gupta,Hyderabad,Mobile,2,26000,Delivered
112,Neha Kappor,Pune,Smartwatch,1,24000,Placed
103,Rohit Mehta,Mumbai,Headphones,5,2200,Delivered
